In [1]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import jaccard

In [2]:
df_meta = pd.read_csv('metadata.txt', sep = '\t', usecols = ['MIT_Accession','Metagenomic_file_name'])
df_meta = df_meta.dropna()
df_meta['Metagenomic_file_name'] = df_meta['Metagenomic_file_name'].str.replace('X', '')
df_meta['Metagenomic_file_name'] = df_meta['Metagenomic_file_name'].str.replace('.', '-')
df_meta = df_meta.rename(columns={'Metagenomic_file_name': 'file'})
df_meta

,MIT_Accession,file
0,24-0019,250728Pat_D25-10264
1,24-0020,250728Pat_D25-10265
2,24-0086,250930Pat_D25-12464
3,24-0087,250930Pat_D25-12457
4,24-0088,250930Pat_D25-12473
...,...,...
85,24-0022,250513Pat_D25-7870
86,24-0023,250513Pat_D25-7871
87,24-0024,250513Pat_D25-7872
88,24-0012,250728Pat_D25-10263


In [3]:
df_mgx = pd.read_csv('nt_prok_blastn_out/nt_prok_blastn_out_taxids.txt', sep = '\t', usecols = ['file','genus'])
df_mgx = df_mgx.drop_duplicates()
df_mgx

,file,genus
0,250513Pat_D25-7849,Rothia
1,250513Pat_D25-7849,Peptostreptococcus
3,250513Pat_D25-7849,Cutibacterium
4,250513Pat_D25-7849,Veillonella
6,250513Pat_D25-7849,Stenotrophomonas
...,...,...
318187,250930Pat_D25-12478,Phytobacter
318213,250930Pat_D25-12478,Haemophilus
318275,250930Pat_D25-12478,Ectopseudomonas
318505,250930Pat_D25-12478,Alcaligenes


In [4]:
dfm = pd.merge(df_mgx,df_meta, on = 'file', how = 'left')
dfm = dfm.drop('file', axis=1)
dfm['method'] = 'mgx'
dfm

,genus,MIT_Accession,method
0,Rothia,23-1068,mgx
1,Peptostreptococcus,23-1068,mgx
2,Cutibacterium,23-1068,mgx
3,Veillonella,23-1068,mgx
4,Stenotrophomonas,23-1068,mgx
...,...,...,...
2429,Phytobacter,24-0116,mgx
2430,Haemophilus,24-0116,mgx
2431,Ectopseudomonas,24-0116,mgx
2432,Alcaligenes,24-0116,mgx


In [5]:
df_culture = pd.read_csv('culture/Culture_metadata_11-2-25_mods_taxids.txt', sep = '\t', usecols = ['MIT_Accession', 'genus'])
df_culture['method'] = 'culture'
df_culture

,MIT_Accession,genus,method
0,23-1062,Cryptobacterium,culture
1,23-1062,Lancefieldella,culture
2,23-1062,Ligilactobacillus,culture
3,23-1062,Rothia,culture
4,23-1062,Rothia,culture
...,...,...,...
1014,24-0133,Streptococcus,culture
1015,24-0133,Streptococcus,culture
1016,24-0133,Streptococcus,culture
1017,24-0133,Streptococcus,culture


In [6]:
dfm = pd.concat([dfm, df_culture], ignore_index=True)
dfm

,genus,MIT_Accession,method
0,Rothia,23-1068,mgx
1,Peptostreptococcus,23-1068,mgx
2,Cutibacterium,23-1068,mgx
3,Veillonella,23-1068,mgx
4,Stenotrophomonas,23-1068,mgx
...,...,...,...
3448,Streptococcus,24-0133,culture
3449,Streptococcus,24-0133,culture
3450,Streptococcus,24-0133,culture
3451,Streptococcus,24-0133,culture


In [7]:
# Create binary table
binary_table = pd.crosstab(
    index=[dfm['MIT_Accession'], dfm['method']],
    columns=dfm['genus']
)

# Convert counts >0 to 1
binary_table = (binary_table > 0).astype(int)

# Reset index to make 'file' and 'method' normal columns
binary_table = binary_table.reset_index()

In [8]:
# Feature columns
feature_cols = binary_table.columns.difference(['genus', 'MIT_Accession', 'method'])

# MIT_Accession with both methods
valid_accessions = binary_table.groupby('MIT_Accession')['method'].nunique()
valid_accessions = valid_accessions[valid_accessions == 2].index

# Filter dataframe
df_filtered = binary_table[binary_table['MIT_Accession'].isin(valid_accessions)]

results = []

for acc in valid_accessions:
    acc_data = df_filtered[df_filtered['MIT_Accession'] == acc]
    culture_row = acc_data[acc_data['method'] == 'culture'][feature_cols].values[0]
    mgx_row = acc_data[acc_data['method'] == 'mgx'][feature_cols].values[0]

    # Skip if both rows are all zeros
    if culture_row.sum() == 0 and mgx_row.sum() == 0:
        continue

    # Jaccard similarity
    jac_sim = 1 - jaccard(culture_row, mgx_row)

    # Shared and unique features
    shared_features = ((culture_row != 0) & (mgx_row != 0)).sum()
    unique_culture = ((culture_row != 0) & (mgx_row == 0)).sum()
    unique_mgx = ((mgx_row != 0) & (culture_row == 0)).sum()

    # Total non-zero features across both methods
    total_nonzero_features = ((culture_row != 0) | (mgx_row != 0)).sum()
    
    # Total features in each method
    total_culture_features = (culture_row != 0).sum()
    total_mgx_features = (mgx_row != 0).sum()

    results.append({
        'MIT_Accession': acc,
        'jaccard_similarity': jac_sim,
        'total_nonzero_features': total_nonzero_features,
        'shared_features': shared_features,
        'unique_culture': unique_culture,
        'unique_mgx': unique_mgx,
        'total_culture_features': total_culture_features,
        'total_mgx_features': total_mgx_features
    })

# Convert to DataFrame
jaccard_df = pd.DataFrame(results)

jaccard_df.to_csv('culture_vs_mgx_stats_genus.txt', sep = '\t', index=False)